# 02 — Feature Audit (ตรวจสอบ Features)

**EN.** Audits the feature matrix produced by `app/ml/forecast/features.py` for data leakage, distribution quality, and missing values. Run this notebook *after* `01_ingest.ipynb` (data must be ingested) and *before* `03_train_baseline.ipynb`. A clean run with all assertions passing is a prerequisite for training.

**TH.** ตรวจสอบ feature matrix ที่สร้างโดย `app/ml/forecast/features.py` สำหรับ data leakage, คุณภาพการกระจาย, และค่าว่าง (missing values). รัน notebook นี้ *หลัง* `01_ingest.ipynb` (ต้องมีข้อมูลแล้ว) และ *ก่อน* `03_train_baseline.ipynb`. การรันผ่านทุก assertion คือเงื่อนไขก่อนเทรน.

**Key invariant / กฎหลัก:** สำหรับทุก row `i` ใน `X` ค่า feature ต้องใช้เฉพาะ observation ที่ `ts_utc <= X.ts_utc[i]` เท่านั้น — ห้ามรั่วอนาคต (no future leakage).

In [ ]:
# Bootstrap — mount repo and add to Python path
# บูตสแตรป — เมาท์ repo และเพิ่ม path สำหรับ Python
import os, sys

REPO_DIR = "/content/heatshield"
if not os.path.exists(REPO_DIR):
    # Fallback: try Heat-wave-backend clone name
    alt = "/content/Heat-wave-backend"
    if os.path.exists(alt):
        REPO_DIR = alt
    else:
        raise RuntimeError(
            f"Repo not found at {REPO_DIR} or {alt}. "
            "Run 00_setup.ipynb first."
        )

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Standard imports
# นำเข้า library มาตรฐาน
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta

matplotlib.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", palette="muted")
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"pandas {pd.__version__}  numpy {np.__version__}  seaborn {sns.__version__}")

In [ ]:
# โหลดข้อมูลตัวอย่างสำหรับ BKK_01 ย้อนหลัง 90 วัน
# Load sample data for BKK_01 — last 90 days relative to DATA_END
from app.data.loaders import read_observations

STATION_ID = "BKK_01"
HORIZON_H = 24
DATA_END = date(2024, 12, 31)
DATA_START = DATA_END - timedelta(days=90)

print(f"Station : {STATION_ID}")
print(f"Window  : {DATA_START}  →  {DATA_END}")

df = read_observations(STATION_ID, DATA_START, DATA_END)

if df.empty:
    raise RuntimeError(
        f"No observations found for {STATION_ID} between {DATA_START} and {DATA_END}. "
        "Run 01_ingest.ipynb first."
    )

# Ensure ts_utc is timezone-aware
# ตรวจสอบให้ ts_utc มี timezone
if not pd.api.types.is_datetime64_any_dtype(df["ts_utc"]):
    df["ts_utc"] = pd.to_datetime(df["ts_utc"], utc=True)

df = df.sort_values("ts_utc").reset_index(drop=True)
print(f"\nLoaded {len(df):,} rows")
print(f"Columns : {list(df.columns)}")
df.head(3)

In [ ]:
# สร้าง feature matrix และแสดง shape + รายชื่อ columns
# Build feature matrix and print shape + column list
from app.ml.forecast.features import build_features

X, y = build_features(df, horizon_h=HORIZON_H)

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"\nFeature columns ({len(X.columns)} total):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:>3d}. {col}")

## 1. Truncation-Invariance Test (No-Leakage)

**EN.** A causal feature pipeline must produce identical values for rows in the past when new future data is appended. The test truncates the dataframe at 80 % of its length, rebuilds features on the truncated version, and compares `_hi_clim` (the climatology column most likely to leak) at shared rows. Maximum absolute difference must be < 1 × 10⁻⁶.

**TH.** pipeline ที่ไม่รั่วอนาคตต้องให้ค่า feature เหมือนเดิมสำหรับ row ในอดีตเมื่อเพิ่มข้อมูลใหม่เข้ามา. การทดสอบนี้ตัด dataframe ที่ 80% ของความยาว สร้าง feature ใหม่ แล้วเปรียบเทียบ `_hi_clim` ใน row ที่ตรงกัน. ความต่างสูงสุดต้องน้อยกว่า 1 × 10⁻⁶.

In [ ]:
# ทดสอบ truncation invariance — ตรวจสอบว่าไม่มี leakage ใน _hi_clim
# Truncation invariance check — verify _hi_clim has no future leakage
from app.ml.forecast.features import build_X_once

cut = int(len(df) * 0.80)
df_full = df.copy()
df_trunc = df.iloc[:cut].copy()

X_full, df_aug_full = build_X_once(df_full)
X_trunc, df_aug_trunc = build_X_once(df_trunc)

# _hi_clim is written onto df_augmented; use ts_utc as the join key
# _hi_clim ถูกเขียนลงใน df_augmented ใช้ ts_utc เป็น key
clim_full = (
    df_aug_full[["ts_utc", "_hi_clim"]]
    .dropna(subset=["_hi_clim"])
    .set_index("ts_utc")
)
clim_trunc = (
    df_aug_trunc[["ts_utc", "_hi_clim"]]
    .dropna(subset=["_hi_clim"])
    .set_index("ts_utc")
)

# Intersect on shared timestamps
# ตัดกันบน timestamp ที่มีร่วมกัน
shared_ts = clim_full.index.intersection(clim_trunc.index)
print(f"Shared rows for comparison: {len(shared_ts):,}")

if len(shared_ts) == 0:
    print("WARNING: No shared rows found — check that df has repeated (station, month, hour) combos.")
else:
    diff = (clim_full.loc[shared_ts, "_hi_clim"] - clim_trunc.loc[shared_ts, "_hi_clim"]).abs()
    max_diff = diff.max()
    print(f"Max |_hi_clim_full - _hi_clim_trunc| = {max_diff:.2e}")
    threshold = 1e-6
    if max_diff < threshold:
        print(f"PASS — truncation invariance holds (max diff {max_diff:.2e} < {threshold:.0e})")
    else:
        bad_rows = diff[diff >= threshold]
        print(f"FAIL — {len(bad_rows)} row(s) exceed threshold. First offenders:")
        print(bad_rows.head(10))
        raise AssertionError("Truncation invariance violated — future data is leaking into past features!")

In [ ]:
# ตรวจสอบ lag column — lag1h ที่ row n ต้องเท่ากับ heat_index_c ที่ row n-1
# Lag column sanity check — hi_lag1h at row n must equal heat_index_c at row n-1
import random

# Need X with ts_utc attached; use df_aug_full to cross-check
# ใช้ df_aug_full เพื่อตรวจสอบค่า lag
X_full_reset, df_aug_chk = build_X_once(df_full)

# Merge ts_utc back onto X_full_reset
# เพิ่ม ts_utc กลับมาใน X เพื่อ join
X_check = X_full_reset.copy()
X_check["ts_utc"] = X_full_reset.attrs["ts_utc"].values

# Build a ts_utc → heat_index_c map from df_aug_chk
# สร้าง map จาก ts_utc → heat_index_c
hi_map = df_aug_chk.set_index("ts_utc")["heat_index_c"]

# Sample 5 rows from the middle of the array (skip first 25 rows — lag NaN zone)
# สุ่ม 5 row จากกลาง array (ข้าม 25 row แรก — โซน NaN ของ lag)
n_rows = len(X_check)
sample_indices = random.sample(range(25, n_rows), min(5, max(0, n_rows - 25)))
sample_indices.sort()

print(f"{'idx':>6}  {'ts_utc':26}  {'hi_lag1h (X)':>14}  {'hi[t-1] (df)':>14}  {'match':>6}")
print("-" * 75)
all_match = True
for idx in sample_indices:
    row = X_check.iloc[idx]
    t_now = row["ts_utc"]
    t_prev = t_now - pd.Timedelta(hours=1)
    lag1_in_X = row["heat_index_c_lag1h"]
    hi_prev = hi_map.get(t_prev, float("nan"))
    match = abs(lag1_in_X - hi_prev) < 1e-6 if not (np.isnan(lag1_in_X) or np.isnan(hi_prev)) else "NaN"
    all_match = all_match and (match is True or match == "NaN")
    print(f"{idx:>6}  {str(t_now):26}  {lag1_in_X:>14.4f}  {hi_prev:>14.4f}  {str(match):>6}")

print()
if all_match:
    print("PASS — hi_lag1h matches heat_index_c[t-1] for all sampled rows.")
else:
    print("FAIL — mismatch detected in lag1h check.")
    raise AssertionError("Lag-1h sanity check failed.")

## 2. Feature Distributions

**EN.** Visual check of lag and rolling feature distributions. Heavy skew or unexpected multi-modal peaks can indicate systematic data quality problems upstream.

**TH.** ตรวจสอบด้วยภาพสำหรับการกระจายของ lag และ rolling features. ความเบ้สูงหรือ peak หลายจุดที่ผิดปกติอาจบ่งชี้ปัญหาคุณภาพข้อมูล.

In [ ]:
# แสดง histogram ของ lag features ทั้งหมด (2-column grid)
# Plot histogram of all lag features in a 2-column grid
lag_cols = [c for c in X.columns if "_lag" in c]
n_lag = len(lag_cols)
n_cols = 2
n_rows_plot = (n_lag + 1) // n_cols

fig, axes = plt.subplots(n_rows_plot, n_cols, figsize=(14, max(4, n_rows_plot * 2.2)))
axes = axes.flatten()

for i, col in enumerate(lag_cols):
    axes[i].hist(X[col].dropna(), bins=40, edgecolor="none", alpha=0.8)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("value", fontsize=8)
    axes[i].set_ylabel("count", fontsize=8)
    axes[i].tick_params(labelsize=7)

# Hide unused axes
# ซ่อน axes ที่ไม่ได้ใช้
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f"Lag Feature Distributions — {STATION_ID} (h={HORIZON_H})", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# แสดง histogram ของ rolling features ทั้งหมด (2-column grid)
# Plot rolling features distributions in a 2-column grid
roll_cols = [c for c in X.columns if "_roll" in c]
n_roll = len(roll_cols)
n_cols = 2
n_rows_plot = (n_roll + 1) // n_cols

fig, axes = plt.subplots(max(1, n_rows_plot), n_cols, figsize=(14, max(4, n_rows_plot * 2.2)))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for i, col in enumerate(roll_cols):
    axes[i].hist(X[col].dropna(), bins=40, edgecolor="none", alpha=0.8, color="steelblue")
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("value", fontsize=8)
    axes[i].set_ylabel("count", fontsize=8)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f"Rolling Feature Distributions — {STATION_ID} (h={HORIZON_H})", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 3. Correlation with Target

**EN.** Pearson correlation of each feature with the h=24 target. Features with very high absolute correlation (> 0.99) may indicate leakage; features near zero may be uninformative dead weight.

**TH.** Pearson correlation ของแต่ละ feature กับ target h=24. Feature ที่มี |correlation| > 0.99 อาจบ่งชี้ leakage; feature ที่ใกล้ศูนย์อาจไม่มีประโยชน์และเป็นน้ำหนักเพิ่มโดยเปล่าประโยชน์.

In [ ]:
# คำนวณ Pearson correlation ของทุก feature กับ y แล้วแสดง top-20 และ bottom-5
# Compute Pearson correlation of every feature with y; show top-20 and bottom-5
combined = X.copy()
combined["__target__"] = y.values
combined_clean = combined.dropna()

corr_series = (
    combined_clean
    .corr(numeric_only=True)["__target__"]
    .drop("__target__", errors="ignore")
    .dropna()
    .sort_values(ascending=False)
)

print(f"Total features with finite correlation: {len(corr_series)}")
print()
print("Top-20 features by |correlation| with target:")
print("-" * 45)
top20 = corr_series.abs().sort_values(ascending=False).head(20)
for feat, val in top20.items():
    signed = corr_series[feat]
    flag = "  *** CHECK LEAKAGE" if abs(val) > 0.99 else ""
    print(f"  {feat:<40s}  {signed:+.4f}{flag}")

print()
print("Bottom-5 features (lowest absolute correlation):")
print("-" * 45)
bottom5 = corr_series.abs().sort_values(ascending=True).head(5)
for feat, val in bottom5.items():
    signed = corr_series[feat]
    print(f"  {feat:<40s}  {signed:+.4f}")

In [ ]:
# Heatmap ของ top-15 feature correlation matrix
# Heatmap of top-15 feature correlation matrix
top15_feats = corr_series.abs().sort_values(ascending=False).head(15).index.tolist()
corr_matrix = combined_clean[top15_feats].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.4,
    ax=ax,
    annot_kws={"size": 7},
)
ax.set_title(
    f"Top-15 Feature Correlation Matrix — {STATION_ID} (h={HORIZON_H})",
    fontsize=11,
    pad=12,
)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)
plt.tight_layout()
plt.show()

## 4. Missing Value Report

**EN.** `build_features` filters rows where any *core* feature is NaN and forward-fills extended/sparse columns. The resulting `X` should be null-free for core columns. Extended columns (ERA5/NASA POWER supplementary fields) may still have NaN — they are imputed at training time using train-split medians.

**TH.** `build_features` กรอง row ที่ core feature เป็น NaN และ forward-fill column แบบ extended/sparse. `X` ที่ได้ควรไม่มีค่าว่างสำหรับ core columns. Extended columns อาจยังมี NaN — จะถูก impute ด้วย median ของ train split ตอนเทรน.

In [ ]:
# นับ null ต่อ feature column และแสดงเป็น table
# Count nulls per feature column and display as table; warn if any core column has nulls
null_counts = X.isnull().sum().rename("null_count")
null_pct = (X.isnull().mean() * 100).rename("null_pct")
null_report = pd.concat([null_counts, null_pct], axis=1)
null_report = null_report[null_report["null_count"] > 0].sort_values("null_count", ascending=False)

if null_report.empty:
    print("No null values found in any feature column.")
    print("PASS — X is null-free.")
else:
    print(f"Columns with nulls ({len(null_report)}):")
    print(null_report.to_string(float_format="{:.2f}".format))
    total_nulls = null_counts.sum()
    # Classify which columns are 'core' vs 'extended'
    # แยก column เป็น 'core' และ 'extended'
    _EXTENDED_PREFIXES = ["solar_wm2", "cloud_cover", "blh_m", "pressure_hpa", "lst_c", "wind_ms", "precip_mm"]
    core_null_cols = [
        c for c in null_report.index
        if not any(c.startswith(p) for p in _EXTENDED_PREFIXES)
    ]
    if core_null_cols:
        print(f"\nWARNING: {len(core_null_cols)} CORE column(s) have nulls: {core_null_cols}")
        raise AssertionError(
            f"Core feature columns contain NaN after build_features: {core_null_cols}. "
            "This should not happen — investigate features.py."
        )
    else:
        print(f"\nAll {len(null_report)} null-bearing column(s) are extended/sparse — OK for training imputation.")

In [ ]:
# ตาราง summary สำหรับทุก station — n_obs, n_train_rows, n_features, null_count, leakage_check
# Summary table across all stations: n_obs, n_train_rows, n_features, null_count, leakage_check
from app.data.stations import STATIONS
from app.ml.forecast.features import build_features, build_X_once

summary_rows = []
for sid in STATIONS:
    row = {"station_id": sid, "n_obs": 0, "n_train_rows": 0, "n_features": 0, "null_count": 0, "leakage_check": "SKIP"}
    try:
        df_s = read_observations(sid, DATA_START, DATA_END)
        if df_s.empty:
            row["leakage_check"] = "NO_DATA"
            summary_rows.append(row)
            continue
        row["n_obs"] = len(df_s)
        X_s, y_s = build_features(df_s, horizon_h=HORIZON_H)
        row["n_train_rows"] = len(X_s)
        row["n_features"] = X_s.shape[1]
        row["null_count"] = int(X_s.isnull().sum().sum())
        # Quick truncation invariance check — compare _hi_clim at cut point
        # ตรวจสอบ truncation invariance เร็ว ๆ
        cut_s = int(len(df_s) * 0.80)
        _, aug_full_s = build_X_once(df_s.copy())
        _, aug_trunc_s = build_X_once(df_s.iloc[:cut_s].copy())
        cf = aug_full_s.set_index("ts_utc")["_hi_clim"]
        ct = aug_trunc_s.set_index("ts_utc")["_hi_clim"]
        shared = cf.index.intersection(ct.index)
        if len(shared) > 0:
            max_d = (cf.loc[shared] - ct.loc[shared]).abs().max()
            row["leakage_check"] = "PASS" if max_d < 1e-6 else f"FAIL(d={max_d:.1e})"
        else:
            row["leakage_check"] = "NO_OVERLAP"
    except Exception as exc:
        row["leakage_check"] = f"ERROR: {exc}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("station_id")
print(summary_df.to_string())

## ✅ Audit Complete — proceed to 03_train_baseline.ipynb

**EN.** If all assertions above passed and the summary table shows `PASS` for every station, the feature pipeline is leak-free and ready for training. Open `03_train_baseline.ipynb` to train the baseline XGBoost forecasters.

**TH.** หาก assertion ทั้งหมดผ่านและตาราง summary แสดง `PASS` สำหรับทุก station แสดงว่า feature pipeline ไม่มี leakage และพร้อมสำหรับการเทรน เปิด `03_train_baseline.ipynb` เพื่อเทรน XGBoost forecaster พื้นฐาน.

---
| Step | Notebook |
|------|----------|
| 1. Setup | `00_setup.ipynb` |
| 2. Ingest | `01_ingest.ipynb` |
| **3. Feature Audit** | **`02_features_audit.ipynb` ← you are here** |
| 4. Train baseline | `03_train_baseline.ipynb` |
| 5. Train quantile | `04_train_quantile.ipynb` |